# symbopt: symbolic and certifying optimization

This tutorial introduces the main mathematical ideas behind `symbopt` and demonstrates the package's public optimization APIs. `symbopt` is a hybrid symbolic/numerical system: it recognizes structure, generates exact or numerical candidates, and then seeks rigorous global evidence when a suitable certificate is available.

The package follows four principles:

1. exploit recognizable structure before using a general method;
2. separate candidate generation from global certification;
3. prefer exact certificates when recoverable;
4. make approximate numerical evidence explicit rather than presenting it as exact proof.

The methods below draw on nonlinear programming, convex optimization, real algebraic geometry, polynomial SOS/moment optimization, and interval analysis.

Project homepage: https://github.com/BhuvaneshBhatt/symbopt


In [ ]:
from pathlib import Path
import sys

# Allow execution from a source checkout without requiring an editable install.
for candidate in (Path.cwd() / "src", Path.cwd().parent / "src"):
    if candidate.exists():
        sys.path.insert(0, str(candidate.resolve()))
        break

import numpy as np
import sympy as sp

from symbopt import (
    minimize, maximize, minval, maxval, argmin, argmax,
    algebraize, algebraize_cases,
    critical_points, build_kkt_system, solve_exact_candidates,
    simplex, revised_simplex, solve_lp, solve_convex_qp,
    SDPProblem, SDPSolverConfig, solve_sdp, solve_sdp_dense,
    sos_decompose, sos_lower_bound,
    sos_scalar_relaxation, sos_poly_relaxation,
    search_sos_hierarchy,
    build_moment_matrix, build_localizing_matrix, build_moment_relaxation,
    extract_moment_atoms, solve_moment_dual,
    link_sos_moments, interval_optimize,
    EvalConfig, arb_available, enclose_expression,
    build_sparsity_graph, build_symmetry_report,
    semialg_available, get_semialg_capabilities,
    cache_info, clear_symbolic_caches,
)


## 1. Algebraization and lifting

Non-polynomial expressions can often be represented by auxiliary variables plus defining equations and domain conditions. This lets downstream polynomial, KKT, semialgebraic, or interval methods reason about a lifted problem. Ordinary polynomial inputs use a fast path and do not create unnecessary lifts.

Fixed exact algebraic numbers such as `RootOf`/`CRootOf` values are preserved as exact algebraic data rather than being converted to floating point.


In [ ]:
x = sp.symbols("x", real=True)
lifted = algebraize(sp.Abs(x - 2) + sp.sqrt(x + 1), [sp.Ge(x, -1)])
print("lifted objective:", lifted.lifted_objective)
print("auxiliary variables:", lifted.auxiliary_variables)
print("defining equalities:", lifted.defining_equalities)


### Piecewise branches

A piecewise objective is expanded into separate branch problems rather than pretending it has a single global polynomial representation.


In [ ]:
cases = algebraize_cases(
    sp.Piecewise((x**2, x >= 0), (-x, True)),
    [sp.Le(x, 2)],
)
for i, case in enumerate(cases, 1):
    print(i, case.original_objective, case.branch_inequalities)


## 2. Symbolic critical points and degeneracy

For smooth unconstrained problems, stationary points satisfy $\nabla f=0$. The Hessian distinguishes ordinary minima, maxima, and saddles. When the Hessian is singular, `symbopt` can inspect nullspace directions using higher-order Taylor information.


In [ ]:
print(critical_points((x - 2)**2, [x]))
print(critical_points(x**4, [x]))


## 3. Constrained KKT candidates

For equalities $g_i(x)=0$ and inequalities $h_j(x)\ge 0$, KKT stationarity combines the objective gradient with constraint gradients, feasibility, and complementary slackness. KKT points are **candidates**; they are not automatically global optima.

The high-level pipeline constructs a KKT system only when a KKT/candidate-search stage is actually needed.


In [ ]:
x, y = sp.symbols("x y", real=True)
p = algebraize(x**2 + y**2, [sp.Eq(x + y, 1), sp.Ge(x, 0), sp.Ge(y, 0)])
kkt = build_kkt_system(p)
print("stationarity:", kkt.stationarity_equations)
print("exact candidates:", solve_exact_candidates(p, kkt))


## 4. High-level optimization API, decision variables, and result semantics

`minimize` and `maximize` return a `PipelineResult` containing candidates, bounds, diagnostics, method provenance, and certificate evidence. Convenience functions return only the certified extremal value or represented optimizers:

- `minval` / `maxval`: certified infimum or supremum when available;
- `argmin` / `argmax`: represented attained optimizer points; they return `()` for an unattained extremum.

Use `variables=` when some symbols are parameters rather than optimization variables. `working_precision=` records the requested precision and routes high-precision inexact inputs away from float64-only fast paths when appropriate.


In [ ]:
x = sp.symbols("x", real=True)
f = (x - 2)**2
print("minval:", minval(f, variables=[x]))
print("argmin:", argmin(f, variables=[x]))
print("maxval on [0, 3]:", maxval(-f, [x >= 0, x <= 3], variables=[x]))
print("argmax on [0, 3]:", argmax(-f, [x >= 0, x <= 3], variables=[x]))

# A symbol not listed in variables= is treated as a parameter.
a = sp.symbols("a", positive=True)
param_result = minimize(a * (x - 1)**2, variables=[x], use_sos=False)
print("decision variables:", param_result.lifted_problem.original_variables)
print("parameters:", param_result.lifted_problem.parameter_symbols)


## 5. Linear programming: simplex, revised simplex, and automatic routing

LP exploits polyhedral structure. The high-level dispatcher recognizes affine objectives and constraints before expensive symbolic or semialgebraic preprocessing, so ordinary LPs are sent directly to the LP solver.


In [ ]:
A = np.array([[1.0, 1.0]])
b = np.array([1.0])
print(simplex(np.array([1.0, 1.0]), A_ub=A, b_ub=b))
print(revised_simplex(np.array([1.0, 1.0]), A_ub=A, b_ub=b))
print(solve_lp(np.array([1.0, 2.0]), A_ub=A, b_ub=b))


## 6. Convex quadratic programming

For $\frac12x^TQx+c^Tx$ with $Q\succeq0$, local optimality conditions are globally sufficient under convex feasibility assumptions. Convex QPs are also routed before general symbolic machinery.


In [ ]:
Q = np.array([[2.0, 0.0], [0.0, 2.0]])
c = np.array([-2.0, -4.0])
qp = solve_convex_qp(Q, c, A_ub=np.array([[1.0, 1.0]]), b_ub=np.array([3.0]))
print(qp)


## 7. Structure-aware global routes

Before invoking general-purpose global machinery, `symbopt` recognizes several exact or certifiable subclasses:

- linear-fractional objectives through Charnes–Cooper transformations and denominator-sign splitting;
- monotone compositions such as exponentials, logarithms, and square roots when domain conditions justify reduction;
- periodic trigonometric expressions, including modest trigonometric polynomials with rationally related frequencies;
- one-variable rational/logarithmic/exponential objectives using domain, pole, limit, and convexity analysis;
- independent-variable components, which can be solved separately and recombined only when every component is certified.


In [ ]:
x, y = sp.symbols("x y", real=True)
frac = minimize((x + 1) / (x + 2), [x >= 0, x <= 3], variables=[x])
print("linear-fractional:", frac.optimum_value, frac.certified)

t = sp.symbols("t", real=True)
trig = minimize(sp.sin(t) + sp.cos(2*t), [t >= 0, t <= 2*sp.pi], variables=[t])
print("trigonometric:", trig.optimum_value, trig.certified)

sep = minimize(sp.exp(x) - 2*x + (y + 1)**2, variables=[x, y], use_sos=False)
print("separable route:", sep.optimum_value, sep.certified)


## 8. Exact semialgebraic backend

When `semialg` is installed, exact polynomial/semialgebraic problems can use its active-set/KKT, RUR, quantifier-elimination, and CAD certification machinery. `semialg_certification` accepts `"auto"`, `"candidate"`, or `"complete"`; automatic mode uses routing heuristics to avoid sending inexpensive polynomial problems into unnecessarily expensive CAD computations.

The backend is optional, so notebooks and applications can inspect its capabilities before relying on it.


In [ ]:
print("semialg available:", semialg_available())
if semialg_available():
    print(get_semialg_capabilities())


## 9. Unified routing and certificates

The top-level result deliberately separates **candidate quality** from **proof quality**. A result may have a good feasible candidate without a rigorous matching global lower bound; conversely, exact or verified routes may certify an optimum directly and skip KKT construction entirely.


In [ ]:
x, y = sp.symbols("x y", real=True)
r = minimize(x**2 + y**2 - 2*x - 4*y,
             [sp.Le(x + y, 3), sp.Ge(x, 0), sp.Ge(y, 0)],
             variables=[x, y])
print("status:", r.status)
print("optimum:", r.optimum_value)
print("attained:", r.attained)
print("certified:", r.certified)
print("KKT built:", r.kkt_system is not None)
print("precision provenance:", r.precision_provenance)


## 10. Semidefinite programming

SDP optimizes a linear objective over positive-semidefinite matrix cones. It is the numerical model underlying Gram-matrix SOS relaxations. Sparse SDP conversion can use chordal clique structure rather than requiring a single dense PSD block.


In [ ]:
C = np.eye(2)
A = [np.array([[1.0, 0.0], [0.0, 0.0]]),
     np.array([[0.0, 0.0], [0.0, 1.0]])]
sdp_problem = SDPProblem.single_block(C=C, A=A, b=np.array([1.0, 2.0]))
sdp = solve_sdp(sdp_problem, SDPSolverConfig(max_iter=100))
print(sdp.status, sdp.primal_objective)


## 11. Exact SOS decomposition

A polynomial SOS certificate has Gram form $p=z^TQz$ with $Q\succeq0$. `symbopt` verifies reconstructed polynomial identities and PSD conditions exactly before labeling a recovered certificate exact. Trivial zero polynomials and recognizable exact squares are handled without an unnecessary numerical SDP solve.


In [ ]:
x, y = sp.symbols("x y", real=True)
p = (x**2 + y**2 + 1)**2 + (x - y)**2
sos = sos_decompose(p, [x, y])
print(sos.success, sos.message)
print("exact certificate:", (sos.certificate.exact_residual_zero and all(v is True for v in sos.certificate.exact_psd_checks)) if sos.certificate else None)
print("zero SOS:", sos_decompose(0, [x]).success)


## 12. SOS global lower bounds and multiplier certificates

For constrained polynomial optimization, a relaxation searches for $\gamma$ and SOS/equality multipliers such that

$$f-\gamma=\sigma_0+\sum_j\sigma_jg_j+\sum_k\tau_kh_k.$$

The SOS–moment bridge is constructed only when SOS or moment machinery was actually used.


In [ ]:
lb, cert = sos_lower_bound((x - 1)**2 + (y + 2)**2 + 3, [x, y])
print("unconstrained lower bound:", lb, "exact SOS:", cert.success)
relax = sos_scalar_relaxation(x**2 + y**2, [x, y], [1 - x**2 - y**2], max_iter=3, multiplier_degree=1)
print(relax.message)


## 13. SOS hierarchy and exact certificate recovery

If a low-order certificate is insufficient, hierarchy search increases the relaxation order. Monomial bases use cached exponent-tuple representations, and sparse/chordal structure is reused where possible. Numerical SDP feasibility alone is not treated as an exact symbolic certificate.


In [ ]:
h = search_sos_hierarchy(x**2, [x], search_lo=-0.1, search_hi=0.1, max_order=1)
print(h.message)
print(h.steps)


## 14. Moment and localizing matrices

Moment relaxations are dual to SOS relaxations. Flat rank structure can reveal a finite atomic measure whose atoms are candidate optimizers.


In [ ]:
x = sp.symbols("x", real=True)
mm = build_moment_matrix([x], 1, {(0,): 1, (1,): 2, (2,): 4})
print(mm.matrix)
loc = build_localizing_matrix(x, [x], 1, {(0,): 1, (1,): 2, (2,): 4, (3,): 8})
print(loc.matrix)


### Dual moment pipeline and SOS–moment consistency


In [ ]:
dual = solve_moment_dual(x**2, [], [x], 1,
                         moment_values={(0,): 1, (1,): 2, (2,): 4})
print(dual.pretty())
consistency = link_sos_moments(x**2, [], [x], dual.candidate_points, order=1)
print(consistency)


## 15. Verified interval optimization and Arb

Interval branch-and-bound bounds the objective over whole boxes rather than isolated sample points. The evaluator uses exact Bernstein bounds for suitable polynomials and, when python-flint is available, Arb ball arithmetic for supported transcendental functions.

Equality-constrained boxes can be contracted before subdivision using interval Newton for scalar equalities and Krawczyk operators for square coupled systems. Trigonometric split information and equality corrections are precompiled and reused.


In [ ]:
t = sp.symbols("t", real=True)
ival = interval_optimize(
    sp.sin(t),
    [sp.Ge(t, 0), sp.Le(t, 2*sp.pi)],
    {t: (0.0, float(2*sp.pi))},
    max_boxes=100,
    tol=1e-3,
)
print(ival.status, ival.best_lower_bound, ival.best_upper_bound)
print("contractions:", ival.contractions_applied, ival.contraction_method)


### Direct rigorous enclosure with Arb

`enclose_expression` exposes the interval evaluator directly. If Arb is unavailable, the package reports that fact rather than silently pretending a transcendental bound is rigorous.


In [ ]:
x = sp.symbols("x", real=True)
print("Arb available:", arb_available())
verified = enclose_expression(x**2, (x,), ((-2, 3),), EvalConfig())
print(verified)


## 16. Sparsity, overlapping chordal decomposition, separability, and symmetry

Polynomial interaction graphs expose which variables interact. Disconnected components can be optimized independently when objective and constraints truly separate. For SOS/SDP models, `symbopt` also supports **overlapping chordal clique decomposition** with consistency constraints on shared separators, so sparse connected graphs need not always become one dense PSD block.

Symmetry diagnostics identify sign and permutation invariances that may support safe reductions.


In [ ]:
x, y, z = sp.symbols("x y z", real=True)
graph = build_sparsity_graph([x*y, y*z], [x, y, z])
print("components:", graph.connected_components())
print(build_symmetry_report(x**2 + y**2, [], [x, y]))


## 17. Symbolic caching and precision provenance

Polynomial construction, monomial bases, gradients, Hessians, quotient reductions, and structural metadata are cached because repeated symbolic work can dominate global optimization. `clear_symbolic_caches()` is useful for benchmarking or controlled experiments.

`PipelineResult.precision_provenance` records the requested/effective precision and whether float64-only backends were eligible.


In [ ]:
clear_symbolic_caches()
_ = critical_points((x - 1)**2 + (y + 1)**2, [x, y])
print(cache_info())

precise = minimize((x - sp.Float("1.25", 60))**2,
                   variables=[x], working_precision=60, use_sos=False)
print(precise.precision_provenance)


## 18. What counts as a proof?

- A KKT point is a **candidate**, not automatically a global optimum.
- A validated convex LP/QP solution has global semantics under its convexity and feasibility assumptions.
- A certified semialgebraic result can prove that no better feasible polynomial point exists; strict domains may yield a certified infimum without attainment.
- An exact SOS Gram identity with exact PSD verification is a symbolic nonnegativity certificate.
- A numerically feasible SDP relaxation is approximate evidence unless exact recovery succeeds.
- An interval lower bound is rigorous only when its evaluator encloses every represented operation on the box.
- `argmin`/`argmax` return no point for a certified but unattained infimum/supremum.

The `OptimalityCertificate`, diagnostics, structural result, semialgebraic result, and interval/SOS fields record which proof artifacts contributed to a conclusion.


## References

- Bertsimas, D., & Tsitsiklis, J. N. (1997). *Introduction to Linear Optimization*. Athena Scientific.
- Blekherman, G., Parrilo, P. A., & Thomas, R. R. (Eds.). (2012). *Semidefinite Optimization and Convex Algebraic Geometry*. SIAM.
- Boyd, S., & Vandenberghe, L. (2004). *Convex Optimization*. Cambridge University Press.
- Cox, D. A., Little, J., & O'Shea, D. (2015). *Ideals, Varieties, and Algorithms* (4th ed.). Springer.
- Lasserre, J. B. (2009). *Moments, Positive Polynomials and Their Applications*. Imperial College Press.
- Laurent, M. (2009). “Sums of Squares, Moment Matrices and Optimization over Polynomials.” In *Emerging Applications of Algebraic Geometry*. Springer.
- Moore, R. E., Kearfott, R. B., & Cloud, M. J. (2009). *Introduction to Interval Analysis*. SIAM.
- Nocedal, J., & Wright, S. J. (2006). *Numerical Optimization* (2nd ed.). Springer.
- Parrilo, P. A. (2000). *Structured Semidefinite Programs and Semialgebraic Geometry Methods in Robustness and Optimization*. PhD thesis, Caltech.
- Rockafellar, R. T. (1970). *Convex Analysis*. Princeton University Press.
